In [0]:
CATALOG = 'SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'SELLERS_ENRICHED'

In [0]:
sellers_df = spark.table('SALES.BRONZE.SELLERS')
sellers_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

sellers_enriched_df = sellers_df.select(

    trim(col("seller_id")).alias("seller_id"), 
    col("seller_zip_code_prefix"), 
    trim(col("seller_city")).alias("seller_city"), 
    col("seller_state"), 
    col("file_name"), 
    col("file_path"), 
    current_timestamp().alias("load_timestamp")).filter(col("seller_id").isNotNull())
    
   
display(sellers_enriched_df.limit(1)) 
     

In [0]:
sellers_enriched_df.createOrReplaceTempView('enriched_sellers_temp_vw')

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY seller_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_sellers_temp_vw
  )
  WHERE rn = 1
) src
ON (src.seller_id  = tgt.seller_id)
WHEN MATCHED THEN
  UPDATE SET
    tgt.seller_id = src.seller_id,
    tgt.seller_zip_code_prefix = src.seller_zip_code_prefix,
    tgt.seller_city = src.seller_city,
    tgt.seller_state = src.seller_state
WHEN NOT MATCHED THEN
  INSERT (
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state,
    file_name,
    origin_file_path,
    load_timestamp
  )
  VALUES (
    src.seller_id,
    src.seller_zip_code_prefix,
    src.seller_city,
    src.seller_state,
    src.file_name,
    src.file_path,
    current_timestamp()
  )
""")


In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{TARGET_TABLE}"))